# Biohub Cell Tracking — Baseline Pipeline

Builds and scores a **classical CV baseline** (3D blob detection + nearest-neighbor
frame-to-frame linking, division-aware) for the cell tracking competition, and
runs a **CPU smoke test** of the official deep-learning baseline.

Reuses the competition host's official companion tooling
([`royerlab/kaggle-cell-tracking-competition`](https://github.com/royerlab/kaggle-cell-tracking-competition),
vendored at `external/kaggle-cell-tracking-competition/`) for the **exact** metric
implementation, `.geff` I/O, and submission-CSV conversion — rather than
reimplementing those (the real metric has several subtleties: duplicate-edge
dedup, merge collapsing, out-degree capping, a +/-1 timepoint division window)
so local scores match the real leaderboard scoring as closely as possible.

Our own code (`src/classical/`) does detection and reuses `tracksdata`'s
built-in nearest-neighbor solver for linking (`max_children=2` natively
handles divisions — no separate division heuristic needed).

No GPU is available on this machine (confirmed during planning), so the DL
baseline section is a tiny smoke test only, not real training.


## 1. Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/mnt/c/Users/sjm57/Desktop/KAGGLE")
VENDOR_ROOT = PROJECT_ROOT / "external" / "kaggle-cell-tracking-competition"
DATA_ROOT = Path("/mnt/d/KAGGLE/data/extracted")

for p in (str(PROJECT_ROOT / "src"), str(VENDOR_ROOT / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

import tracksdata as td
import torch
import zarr
from geff import GeffMetadata

from tracking_cellmot.io import save_graph
from tracking_cellmot.metrics import evaluate as compute_metric, node_recall, per_sample_metrics, summarise

from classical.pipeline import run_sample

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("zarr", zarr.__version__, "| tracksdata", td.__version__)
print("data root exists:", DATA_ROOT.exists())


torch 2.13.0+cpu | cuda available: False
zarr 3.1.6 | tracksdata 0.1.dev1+g39dccf3a2
data root exists: True


## 2. Train/validation split

Stratified 80/20 split by dataset-source prefix (`44b6`/`6bba`), generated by `src/data_split.py` and written in the exact schema `train_unet_transformer.py --splits` expects, so a future full DL training run can reuse this identical held-out set.

In [2]:
splits_path = DATA_ROOT / "train" / "dataset_splits.json"
folds = json.loads(splits_path.read_text())
fold0 = folds[0]
print(f"train: {len(fold0['train'])} samples | held-out val: {len(fold0['test'])} samples")

# Local disk only has a *subsample* of the val fold extracted (full-archive
# extraction of all 199 samples was impractically slow over this mount -- see
# below). This subsample is enough for a meaningful, unbiased score estimate;
# re-running scripts/extract_targeted-style code with more ids would extend it.
val_dir = DATA_ROOT / "train"
val_sample_ids = sorted(p.stem for p in val_dir.glob("*.zarr"))
print(f"extracted val subsample: {len(val_sample_ids)} samples")
print(val_sample_ids)


train: 159 samples | held-out val: 40 samples
extracted val subsample: 20 samples
['44b6_0db75fae', '44b6_341df25f', '44b6_668e0cc7', '44b6_706092f0', '44b6_7a302da0', '44b6_808952d6', '44b6_9be80b04', '44b6_aaf8b0ea', '44b6_abf82518', '44b6_c8e2a523', '6bba_2312ac41', '6bba_23af9eeb', '6bba_2646afc7', '6bba_3a1849c2', '6bba_3abfe10a', '6bba_3fda6b25', '6bba_55b7eebe', '6bba_6479435d', '6bba_6db0e9b4', '6bba_718b21f9']


## 3. Classical baseline on the validation subsample

Runs detection + nearest-neighbor linking on each extracted validation sample, saving predicted `.geff`s.

In [3]:
pred_val_dir = PROJECT_ROOT / "predictions" / "classical" / "val"
pred_val_dir.mkdir(parents=True, exist_ok=True)

summaries = []
for sid in val_sample_ids:
    t0 = time.time()
    zarr_path = val_dir / f"{sid}.zarr"
    out_path = pred_val_dir / f"{sid}.geff"
    summary = run_sample(zarr_path, out_path)
    summary["elapsed_s"] = round(time.time() - t0, 1)
    summaries.append(summary)
    print(summary)

summary_df = pd.DataFrame(summaries)
summary_df


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[07/23/26 22:43:16] WARNING  No nodes found for time point in range (-1 <= t < 0)            ]8;id=12140611;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=12140612;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py#318\318]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 563.80it/s]


{'sample_id': '44b6_0db75fae', 'n_detections': 10347, 'n_nodes': 10022, 'n_edges': 9286, 'elapsed_s': 50.5}


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

...

Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[07/23/26 22:58:48] WARNING  No nodes found for time point in range (-1 <= t < 0)            ]8;id=12140725;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=12140726;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py#318\318]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 732.92it/s]


{'sample_id': '6bba_718b21f9', 'n_detections': 4580, 'n_nodes': 4474, 'n_edges': 4258, 'elapsed_s': 49.1}


,sample_id,n_detections,n_nodes,n_edges,elapsed_s
0,44b6_0db75fae,10347,10022,9286,50.5
1,44b6_341df25f,5787,5678,5423,48.9
2,44b6_668e0cc7,29115,28666,27311,49.1
3,44b6_706092f0,39247,38446,36981,49.4
4,44b6_7a302da0,37362,36462,34676,45.3
5,44b6_808952d6,12268,12029,11493,46.0
6,44b6_9be80b04,35420,34576,32963,52.8
7,44b6_aaf8b0ea,12798,12289,11479,47.9
8,44b6_abf82518,20084,19588,18411,50.6
9,44b6_c8e2a523,28274,27912,26920,49.9


## 4. Official local scoring

Scores our predicted `.geff`s against the real ground-truth `.geff`s using the vendored `tracking_cellmot.metrics` implementation directly (same code path as `scripts/evaluate.py`), so results reflect the real competition formula: `score = adj_edge_jaccard + 0.1 * division_jaccard`, with `adj_edge_jaccard = max(0, jaccard * (1 - 0.1*(N_pred-N_true)/N_true))` using each sample's real `estimated_number_of_nodes` geff metadata as `N_true`.

In [4]:
def load_graph(path):
    result = td.graph.IndexedRXGraph.from_geff(path)
    return result[0] if isinstance(result, tuple) else result


SCALE = (1.625, 0.40625, 0.40625)

rows = []
for sid in val_sample_ids:
    pred_graph = load_graph(pred_val_dir / f"{sid}.geff")
    gt_path = val_dir / f"{sid}.geff"
    gt_graph = load_graph(gt_path)

    er = compute_metric(pred_graph, gt_graph, scale=SCALE, max_distance=7.0)
    recall = node_recall(pred_graph, gt_graph) if pred_graph.num_edges() > 0 else 0.0
    n_total = (GeffMetadata.read(gt_path).extra or {}).get("estimated_number_of_nodes", float("nan"))

    row = per_sample_metrics(er, n_total, recall)
    row["sample_id"] = sid
    rows.append(row)

per_sample_df = pd.DataFrame(rows)
per_sample_df


[07/23/26 22:59:23] WARNING  No matching nodes found.                                           ]8;id=12140733;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/graph/_base_graph.py\_base_graph.py]8;;\:]8;id=12140734;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/graph/_base_graph.py#1256\1256]8;;\

[07/23/26 22:59:52] WARNING  No matching nodes found.                                           ]8;id=12140739;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/graph/_base_graph.py\_base_graph.py]8;;\:]8;id=12140740;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/graph/_base_graph.py#1256\1256]8;;\

,edge_tp,edge_fp,edge_fn,division_tp,division_fp,division_fn,num_pred_nodes,node_recall,total_node_ratio,edge_jaccard,adj_edge_jaccard,sample_id
0,141,6,10,0,4,0,10022,0.942675,-0.346462,0.898089,0.929205,44b6_0db75fae
1,191,8,18,1,5,0,5678,0.921296,-0.236417,0.880184,0.900993,44b6_341df25f
2,551,71,80,0,33,0,28666,0.931571,-0.277953,0.784900,0.806717,44b6_668e0cc7
3,1,0,116,0,0,1,38446,0.016949,-0.449970,0.008547,0.008932,44b6_706092f0
4,277,151,230,1,33,0,36462,0.718992,-0.454635,0.420973,0.440112,44b6_7a302da0
5,172,14,7,0,8,0,12029,0.994475,-0.235866,0.891192,0.912212,44b6_808952d6
6,154,49,66,0,21,1,34576,0.860360,-0.402595,0.572491,0.595539,44b6_9be80b04
7,1,0,205,0,0,1,12289,0.009569,-0.443911,0.004854,0.005070,44b6_aaf8b0ea
8,90,3,4,0,2,0,19588,0.968750,-0.290393,0.927835,0.954779,44b6_abf82518
9,145,20,19,0,3,1,27912,0.958333,-0.157857,0.788043,0.800483,44b6_c8e2a523


In [5]:
s = summarise(rows)
print("=== Classical baseline: validation subsample score ===")
print(f"n={s['n']}  score={s['score']:.4f}")
print(f"edge_jaccard={s['edge_jaccard']:.4f}  adj_edge_jaccard={s['adj_edge_jaccard']:.4f} (n_adj={s['n_adj']})")
print(f"division_jaccard={s['division_jaccard']:.4f}  (TP={s['division_tp']} FP={s['division_fp']} FN={s['division_fn']})")
print(f"node_recall={s['node_recall']:.4f}")


=== Classical baseline: validation subsample score ===
n=20  score=0.5371
edge_jaccard=0.5227  adj_edge_jaccard=0.5368 (n_adj=20)
division_jaccard=0.0031  (TP=2 FP=641 FN=9)
node_recall=0.7021


## 5. Classical baseline on the test set -> submission.csv

Runs the identical pipeline on all 4 real test volumes (no ground truth), then converts the predicted `.geff`s to `submission.csv` with the vendored `scripts/geffs_to_csv.py` logic.

In [6]:
test_dir = DATA_ROOT / "test"
test_sample_ids = sorted(p.stem for p in test_dir.glob("*.zarr"))
print("test samples:", test_sample_ids)

pred_test_dir = PROJECT_ROOT / "predictions" / "classical" / "test"
pred_test_dir.mkdir(parents=True, exist_ok=True)

test_summaries = []
for sid in test_sample_ids:
    t0 = time.time()
    summary = run_sample(test_dir / f"{sid}.zarr", pred_test_dir / f"{sid}.geff")
    summary["elapsed_s"] = round(time.time() - t0, 1)
    test_summaries.append(summary)
    print(summary)

pd.DataFrame(test_summaries)


test samples: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[07/23/26 23:02:00] WARNING  No nodes found for time point in range (-1 <= t < 0)            ]8;id=12140745;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=12140746;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py#318\318]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 423.27it/s]


{'sample_id': '44b6_0113de3b', 'n_detections': 23054, 'n_nodes': 22818, 'n_edges': 21840, 'elapsed_s': 50.1}


...

Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[07/23/26 23:04:45] WARNING  No nodes found for time point in range (-1 <= t < 0)            ]8;id=12140788;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=12140789;file:///mnt/c/Users/sjm57/Desktop/KAGGLE/external/tracksdata/src/tracksdata/edges/_distance_edges.py#318\318]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 295.06it/s]


{'sample_id': '6bba_05db0fb1', 'n_detections': 39837, 'n_nodes': 39017, 'n_edges': 36904, 'elapsed_s': 50.4}


,sample_id,n_detections,n_nodes,n_edges,elapsed_s
0,44b6_0113de3b,23054,22818,21840,50.1
1,44b6_0b24845f,8801,8092,7071,60.4
2,6bba_05b6850b,5335,5199,4833,56.6
3,6bba_05db0fb1,39837,39017,36904,50.4


In [7]:
sys.path.insert(0, str(VENDOR_ROOT / "scripts"))
from geffs_to_csv import geffs_to_csv

submission_path = PROJECT_ROOT / "submissions" / "submission.csv"
submission_path.parent.mkdir(parents=True, exist_ok=True)
geffs_to_csv(pred_test_dir, submission_path)

sub_df = pd.read_csv(submission_path)
print(sub_df.shape)
print("datasets covered:", sorted(sub_df["dataset"].unique()))
print("expected (from test dir):", test_sample_ids)
assert set(sub_df["dataset"].unique()) == set(test_sample_ids), "submission must cover every test dataset"
sub_df.head()


44b6_0113de3b: 22818 nodes, 21840 edges
44b6_0b24845f: 8092 nodes, 7071 edges
6bba_05b6850b: 5199 nodes, 4833 edges
6bba_05db0fb1: 39017 nodes, 36904 edges

Wrote 145774 rows from 4 geffs to /mnt/c/Users/sjm57/Desktop/KAGGLE/submissions/submission.csv
(145774, 10)
datasets covered: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']
expected (from test dir): ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']


,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,0,0,41,7,172,-1,-1
1,1,44b6_0113de3b,node,1,0,20,179,83,-1,-1
2,2,44b6_0113de3b,node,2,0,17,104,84,-1,-1
3,3,44b6_0113de3b,node,4,0,48,100,171,-1,-1
4,4,44b6_0113de3b,node,5,0,58,136,202,-1,-1


## 6. DL baseline: CPU smoke test

The official DL baseline (3D U-Net + temporal cross-attention transformer, `scripts/train_unet_transformer.py`) needs real multi-epoch training to produce usable predictions. This machine has no GPU, so this cell only runs a **tiny 1-epoch / 1-iteration smoke test on a single debug video** to confirm the code path executes correctly end-to-end on CPU -- it is not expected to produce a good model. Real training belongs on Kaggle's GPU notebook environment (<=12h budget) or another GPU machine, as a follow-up task.

Note: the vendored script calls `torch.cuda.synchronize()` unconditionally at a few points (meant as GPU timing barriers); `src/dl_smoke_test.py` shims it to a no-op in-process when no CUDA device is present, rather than patching the vendored file, so the real training/detection/edge-prediction code runs unmodified.

In [8]:
from dl_smoke_test import run_smoke_test

debug_video = val_dir / val_sample_ids[0]
print(f"Smoke-testing on: {debug_video.name}")

t0 = time.time()
run_smoke_test(VENDOR_ROOT, debug_video, epochs=1, max_iters=1)
print(f"\nSmoke test finished in {time.time()-t0:.1f}s (this is a CPU code-path check, not real training).")


Smoke-testing on: 44b6_0db75fae
Debug mode: using single video 44b6_0db75fae
Loading train (1 datasets)...


train: 100%|██████████| 1/1 [00:04<00:00,  4.04s/it]

  train done: 88 windows total
Loading test (1 datasets)...



test: 100%|██████████| 1/1 [00:01<00:00,  1.10s/it]

  test done: 88 windows total
max_nodes=2
Using device: cpu | visible CUDA GPUs: 0
Model parameters: 2,076,706


...

Training:   0%|          | 0/1 [10:42<?, ?it/s, acc=0.0000, det=0.9200, edge=0.0000]

  Epoch   0/1 | edge=0.0000 | det=0.9200 | test_loss=0.0000 | acc=0.0000 | recall=0.0000 | best=0.0000 * | train=28.5s test=614.1s


Training: 100%|██████████| 1/1 [10:42<00:00, 642.89s/it, acc=0.0000, det=0.9200, edge=0.0000]


Best score (acc*recall): 0.0000, saved to /mnt/c/Users/sjm57/Desktop/KAGGLE/external/kaggle-cell-tracking-competition/weights/unet_transformer/split_0/edge_predictor_best.pth



Smoke test finished in 651.0s (this is a CPU code-path check, not real training).


## 7. Summary

- **Classical baseline**: a real, working detect -> link -> divide -> `.geff` -> `submission.csv`
  pipeline, scored locally with the *official* metric implementation on a held-out
  validation subsample (see section 4 for the actual numbers this run produced).
  This is the concrete deliverable of this notebook.
- **DL baseline**: wired up and confirmed to run end-to-end on CPU (section 6), but not
  meaningfully trained here — no GPU locally. Next step: run
  `scripts/train_unet_transformer.py --split 0 --epochs <N>` (using the
  `dataset_splits.json` fold generated in section 2) on Kaggle's GPU notebook or
  another GPU machine, then `scripts/predict_unet_transformer.py` +
  `scripts/geffs_to_csv.py` to produce a DL submission, and compare its
  `evaluate.py` score against the classical baseline's score above.
